In [1]:
import pandas as pd
import numpy as np

In [2]:
# Set random seed for reproducibility
np.random.seed(42)

In [3]:
# Contextual detail
bank_name = "Sunrise Microfinance Bank"

In [4]:
# Nigerian surnames and cities for localization
nigerian_surnames = [
    'Adebayo', 'Okafor', 'Chukwu', 'Bello', 'Adekunle', 'Okonkwo', 'Abdullahi', 
    'Ibrahim', 'Eze', 'Olawale', 'Nwachukwu', 'Yusuf', 'Adesina', 'Onuoha', 'Sule'
]
nigerian_cities = ['Lagos', 'Abuja', 'Port Harcourt', 'Kano', 'Ibadan', 'Enugu']

In [5]:
# Generate synthetic data
n_customers = 1000
data = {
    'CustomerId': [f'SMB{cid:06d}' for cid in range(15565701, 15565701 + n_customers)],  # Unique IDs
    'Surname': np.random.choice(nigerian_surnames, n_customers),
    # Demographic Attributes
    'Age': np.random.randint(18, 92, n_customers),  # Age range
    'Gender': np.random.choice(['Male', 'Female'], n_customers, p=[0.55, 0.45]),
    'Marital Status': np.random.choice(['Single', 'Married', 'Divorced'], n_customers, p=[0.4, 0.5, 0.1]),
    'Education Level': np.random.choice(['None', 'Primary', 'Secondary', 'Tertiary'], n_customers, p=[0.2, 0.3, 0.3, 0.2]),
    # Account & Transaction Attributes
    'Account Balance Trend': np.random.choice(['Stable', 'Increasing', 'Decreasing'], n_customers, p=[0.5, 0.3, 0.2]),
    'Loan History': np.random.choice(['Active', 'Defaulted', 'Cleared'], n_customers, p=[0.4, 0.2, 0.4]),
    'Frequency of Deposits/Withdrawals': np.random.poisson(5, n_customers),  # Transactions per month
    'Average Transaction Value': np.random.uniform(1000, 50000, n_customers),  # Transaction value in NGN
    'Account Activity': np.random.choice(['Active', 'Dormant'], n_customers, p=[0.6, 0.4]),
    # Service Utilization Attributes
    'Use of Savings Products': np.random.choice([0, 1], n_customers, p=[0.4, 0.6]),  # 0: No, 1: Yes
    'Use of Loan Products': np.random.choice([0, 1], n_customers, p=[0.3, 0.7]),  # 0: No, 1: Yes
    'Use of Digital Banking': np.random.choice(['None', 'USSD', 'App', 'Both'], n_customers, p=[0.2, 0.5, 0.2, 0.1]),
    'Participation in Group Lending': np.random.choice([0, 1], n_customers, p=[0.5, 0.5]),  # 0: No, 1: Yes
    # Customer Relationship Attributes
    'Tenure': np.random.randint(0, 10, n_customers),  # Years with bank
    'Number of Complaints Logged': np.random.randint(0, 5, n_customers),  # Complaints filed
    'Response Time to Complaints': np.random.randint(0, 15, n_customers),  # Days to resolve complaints
    'Customer Support Interactions': np.random.randint(0, 10, n_customers),  # Support interactions
    # Payment Behavior
    'Repayment Timeliness': np.random.choice(['On-time', 'Late'], n_customers, p=[0.7, 0.3]),
    'Overdue Loan Frequency': np.random.randint(0, 5, n_customers),  # Number of overdue loans
    'Penalties Paid': np.random.uniform(0, 10000, n_customers)  # Penalties in NGN
}

In [6]:
# Create initial churn label (Exited) based on behavioral patterns
data['Exited'] = [
    1 if (data['Account Balance Trend'][i] == 'Decreasing' or 
          data['Loan History'][i] == 'Defaulted' or 
          data['Frequency of Deposits/Withdrawals'][i] < 2 or 
          data['Account Activity'][i] == 'Dormant' or 
          data['Number of Complaints Logged'][i] > 3 or 
          data['Response Time to Complaints'][i] > 10 or 
          data['Repayment Timeliness'][i] == 'Late' or 
          data['Overdue Loan Frequency'][i] > 3 or 
          data['Penalties Paid'][i] > 5000 or 
          np.random.random() < 0.05) else 0
    for i in range(n_customers)
]

In [7]:
# Adjust to achieve ~20% churn rate
churn_indices = np.random.choice(n_customers, size=int(0.2 * n_customers), replace=False)
data['Exited'] = [1 if i in churn_indices else 0 for i in range(n_customers)]

# Create DataFrame
df = pd.DataFrame(data)



In [8]:

# Data Preprocessing
# 1. Data Cleaning
# Handle missing values
df.fillna({
    'Age': df['Age'].median(),
    'Tenure': df['Tenure'].median(),
    'Frequency of Deposits/Withdrawals': df['Frequency of Deposits/Withdrawals'].median(),
    'Average Transaction Value': df['Average Transaction Value'].median(),
    'Number of Complaints Logged': df['Number of Complaints Logged'].median(),
    'Response Time to Complaints': df['Response Time to Complaints'].median(),
    'Customer Support Interactions': df['Customer Support Interactions'].median(),
    'Overdue Loan Frequency': df['Overdue Loan Frequency'].median(),
    'Penalties Paid': df['Penalties Paid'].median(),
    'Gender': df['Gender'].mode()[0],
    'Marital Status': df['Marital Status'].mode()[0],
    'Education Level': df['Education Level'].mode()[0],
    'Account Balance Trend': df['Account Balance Trend'].mode()[0],
    'Loan History': df['Loan History'].mode()[0],
    'Account Activity': df['Account Activity'].mode()[0],
    'Use of Savings Products': df['Use of Savings Products'].mode()[0],
    'Use of Loan Products': df['Use of Loan Products'].mode()[0],
    'Use of Digital Banking': df['Use of Digital Banking'].mode()[0],
    'Participation in Group Lending': df['Participation in Group Lending'].mode()[0],
    'Repayment Timeliness': df['Repayment Timeliness'].mode()[0]
}, inplace=True)

# Remove duplicates
df.drop_duplicates(subset='CustomerId', keep='first', inplace=True)



In [9]:
df

,CustomerId,Surname,Age,Gender,Marital Status,Education Level,Account Balance Trend,Loan History,Frequency of Deposits/Withdrawals,Average Transaction Value,...,Use of Digital Banking,Participation in Group Lending,Tenure,Number of Complaints Logged,Response Time to Complaints,Customer Support Interactions,Repayment Timeliness,Overdue Loan Frequency,Penalties Paid,Exited
0,SMB15565701,Abdullahi,18,Male,Single,Primary,Stable,Active,5,1406.937215,...,USSD,1,9,2,11,1,On-time,1,5458.300097,0
1,SMB15565702,Bello,68,Male,Single,Tertiary,Increasing,Defaulted,7,48532.810642,...,USSD,0,5,1,3,2,On-time,4,2868.064371,1
2,SMB15565703,Adesina,62,Male,Married,Tertiary,Increasing,Cleared,3,22153.445954,...,App,0,2,3,13,0,Late,4,2501.341043,0
3,SMB15565704,Sule,21,Male,Single,Primary,Stable,Defaulted,5,48498.131107,...,Both,0,4,2,2,1,On-time,4,47.661305,0
4,SMB15565705,Nwachukwu,79,Male,Married,Tertiary,Decreasing,Cleared,7,7875.852469,...,Both,1,0,4,11,1,On-time,2,3966.103421,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,SMB15566696,Adebayo,53,Female,Married,Secondary,Increasing,Cleared,1,32180.871451,...,USSD,0,9,2,3,9,On-time,0,2580.192627,0
996,SMB15566697,Adekunle,62,Male,Single,Tertiary,Stable,Defaulted,7,25504.524453,...,USSD,1,3,2,9,2,On-time,3,3506.360815,0
997,SMB15566698,Onuoha,40,Female,Married,None,Decreasing,Cleared,5,6061.723413,...,App,1,4,1,8,4,On-time,4,9158.745673,0
998,SMB15566699,Eze,64,Female,Single,None,Increasing,Active,7,10684.741275,...,USSD,0,9,4,11,7,Late,1,3914.977383,0


In [10]:
# Summary statistics
print(f"Transactional Data Summary for {bank_name}:")
print(df.describe(include='all'))
print("\nChurn Distribution:")
print(df['Exited'].value_counts(normalize=True))

Transactional Data Summary for Sunrise Microfinance Bank:
         CustomerId Surname          Age Gender Marital Status  \
count          1000    1000  1000.000000   1000           1000   
unique         1000      15          NaN      2              3   
top     SMB15565701   Yusuf          NaN   Male        Married   
freq              1      87          NaN    545            495   
mean            NaN     NaN    53.798000    NaN            NaN   
std             NaN     NaN    21.221912    NaN            NaN   
min             NaN     NaN    18.000000    NaN            NaN   
25%             NaN     NaN    35.000000    NaN            NaN   
50%             NaN     NaN    54.000000    NaN            NaN   
75%             NaN     NaN    72.000000    NaN            NaN   
max             NaN     NaN    91.000000    NaN            NaN   

       Education Level Account Balance Trend Loan History  \
count             1000                  1000         1000   
unique               4     

In [11]:
df.to_csv(f'synthetic_data_{bank_name.replace(" ", "_").lower()}.csv', index=False)